# Async Subagents

In the basics notebook, a supervisor delegated with the **`task`** tool. That call is **synchronous**: the supervisor stops and waits, idle, until the subagent returns its final report.

An **async subagent** runs as a **background job on an [Agent Protocol](https://github.com/langchain-ai/agent-protocol) server**. Launching one returns a **task id immediately** — the supervisor keeps talking to you while the work happens elsewhere. You can then **check**, **update**, or **cancel** the job by id.

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead><tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">&nbsp;</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Sync subagent (<code>task</code>)</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Async subagent</th>
</tr></thead>
<tbody>
<tr><td style="border:1px solid #999; padding:10px;"><b>Launch</b></td><td style="border:1px solid #999; padding:10px;">Blocks until the subagent finishes</td><td style="border:1px solid #999; padding:10px;">Returns a task id immediately</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Concurrency</b></td><td style="border:1px solid #999; padding:10px;">Parallel, but the supervisor is blocked</td><td style="border:1px solid #999; padding:10px;">Parallel and non-blocking</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Mid-task updates</b></td><td style="border:1px solid #999; padding:10px;">Not possible</td><td style="border:1px solid #999; padding:10px;"><code>update_async_task</code></td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Cancellation</b></td><td style="border:1px solid #999; padding:10px;">Not possible</td><td style="border:1px solid #999; padding:10px;"><code>cancel_async_task</code></td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Runs</b></td><td style="border:1px solid #999; padding:10px;">In-process, ephemeral</td><td style="border:1px solid #999; padding:10px;">On a server, on its own thread</td></tr>
<tr><td style="border:1px solid #999; padding:10px;"><b>Best for</b></td><td style="border:1px solid #999; padding:10px;">Short work you wait on</td><td style="border:1px solid #999; padding:10px;">Long-running, interactive work</td></tr>
</tbody></table>

> Async subagents are a **preview** feature. They talk to any Agent Protocol server — a LangSmith Deployment or a self-hosted one.

## How it works

The supervisor and the subagent live in **two different processes**:

- The **supervisor** runs right here in the notebook (`create_deep_agent`).
- The **`researcher` subagent** is served by a local **`langgraph dev`** server that speaks the Agent Protocol.

`create_deep_agent` sees a subagent spec that carries a **`graph_id`** and routes it to **`AsyncSubAgentMiddleware`** instead of the usual `task` tool. That middleware adds **five tools** — `start_async_task`, `check_async_task`, `update_async_task`, `cancel_async_task`, `list_async_tasks` — and keeps a dedicated **`async_tasks`** channel in the supervisor's state so task ids survive context compaction.

**Transports:** omit `url` and calls route in-process (**ASGI**, for co-deployed graphs). Set `url` and calls go over **HTTP** to a remote (or, as here, a local) server. We use HTTP to `localhost:2024`.

## Step 1 — Start the subagent server

Two files in this repo define the deployment:

- **`async_agents/researcher.py`** — an ordinary compiled deep agent (Tavily search + a travel-research prompt) exposed as `graph`.
- **`langgraph.json`** — registers it as graph id `researcher` and points the server at `.env` for API keys.

In a **separate terminal**, start the server and leave it running:

```bash
uv run langgraph dev --no-browser --n-jobs-per-worker 10
```

`--n-jobs-per-worker 10` widens the worker pool so several background tasks can run at once (the default is 1). The cell below checks that the server is reachable.

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
from dotenv import load_dotenv
load_dotenv(override=True)

import time
from util import print_last_exchange
from langgraph.checkpoint.memory import MemorySaver
from langgraph_sdk import get_sync_client
from deepagents import AsyncSubAgent, create_deep_agent

model = "claude-haiku-4-5-20251001"
SERVER_URL = "http://localhost:2024"

In [29]:
# The server must be running (see Step 1). This confirms it's up and the graph is registered.
client = get_sync_client(url=SERVER_URL)
print("Registered graphs:", [a["graph_id"] for a in client.assistants.search()])

Registered graphs: ['researcher']


## Step 2 — Wire the supervisor to the async subagent

An `AsyncSubAgent` is just a spec: a `name`, a `description` the supervisor uses to decide when to delegate, the `graph_id` on the server, and an optional `url` (present here, so HTTP transport).

We give the supervisor a **checkpointer + `thread_id`** so its `async_tasks` channel persists across turns — that's how it remembers task ids between our messages.

In [30]:
researcher = AsyncSubAgent(
    name="researcher",
    description=(
        "Researches a travel destination or question using web search. "
        "Use for questions that need multiple searches and synthesis."
    ),
    graph_id="researcher",   # must match a graph in langgraph.json
    url=SERVER_URL,          # url present -> HTTP transport to the server
)

supervisor = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel-planning supervisor. Delegate research to your async "
        "`researcher` subagent, and relay its findings back to the user."
    ),
    subagents=[researcher],
    checkpointer=MemorySaver(),   # persists the async_tasks channel across turns
)

config = {"configurable": {"thread_id": "trip-planning"}}

In [31]:
# Because the spec carried a graph_id, create_deep_agent wired AsyncSubAgentMiddleware,
# which contributes these five tools for managing background work:
from deepagents import AsyncSubAgentMiddleware

for t in AsyncSubAgentMiddleware(async_subagents=[researcher]).tools:
    print("-", t.name)

- start_async_task
- check_async_task
- update_async_task
- cancel_async_task
- list_async_tasks


## Launch a background task

We ask the supervisor to kick off research **in the background**. Notice the reply comes back right away with a **task id** — the supervisor does *not* sit and wait for the research, and it won't poll on its own.

In [32]:
result = supervisor.invoke({"messages": [{"role": "user", "content": (
    "Research the best time of year to visit Kyoto, Japan. "
    "Kick it off in the background and just give me the task id."
)}]}, config=config)
print_last_exchange(result)

### Human

Research the best time of year to visit Kyoto, Japan. Kick it off in the background and just give me the task id.

### Ai

Task ID: `019fd422-7840-78f1-8096-742acbc7fc3e`

I've started the research in the background. Let me know when you'd like the results.

### The task lives in the `async_tasks` state channel

The task id isn't only buried in a tool message — the middleware records it in a dedicated state channel, so it survives even when the conversation history is summarized.

In [33]:
def show_tasks():
    tasks = supervisor.get_state(config).values.get("async_tasks", {})
    for tid, t in tasks.items():
        print(f"{tid}  status={t['status']:9}  agent={t['agent_name']}")
    return tasks

tasks = show_tasks()

019fd422-7840-78f1-8096-742acbc7fc3e  status=running    agent=researcher


## Run several at once

Because launching is non-blocking, the supervisor can fan out **multiple** background jobs from a single turn.

In [34]:
result = supervisor.invoke({"messages": [{"role": "user", "content": (
    "Also research two more things in the background: "
    "(1) which neighborhoods to stay in when visiting Kyoto, and "
    "(2) good day trips from Kyoto. Launch both and report the task ids."
)}]}, config=config)
print_last_exchange(result)
tasks = show_tasks()

### Human

Also research two more things in the background: (1) which neighborhoods to stay in when visiting Kyoto, and (2) good day trips from Kyoto. Launch both and report the task ids.

### Ai

Task IDs:
1. Best neighborhoods: `019fd422-888f-74a3-97ae-0bf1bef830ed`
2. Day trips from Kyoto: `019fd422-8890-7942-b7bb-5a6cf16011f0`

Both are running in the background now.

019fd422-7840-78f1-8096-742acbc7fc3e  status=running    agent=researcher
019fd422-888f-74a3-97ae-0bf1bef830ed  status=running    agent=researcher
019fd422-8890-7942-b7bb-5a6cf16011f0  status=running    agent=researcher


## Check status — live

`list_async_tasks` fetches the **current** status of every tracked task from the server (statuses saved in the chat history are always considered stale).

The background jobs take a little while, so we first poll the server directly until they finish — purely so this notebook renders completed results. In normal use you'd just carry on with other work and ask for an update whenever you like.

In [35]:
def wait_all(task_map, timeout=240):
    deadline = time.time() + timeout
    for t in task_map.values():
        while time.time() < deadline:
            run = client.runs.get(thread_id=t["thread_id"], run_id=t["run_id"])
            if run["status"] != "running":
                break
            time.sleep(3)

wait_all(tasks)

result = supervisor.invoke({"messages": [{"role": "user", "content":
    "List all my research tasks and their current status."
}]}, config=config)
print_last_exchange(result)

### Human

List all my research tasks and their current status.

### Ai

All three tasks are complete:

1. **Best time to visit Kyoto** (`019fd422-7840-78f1-8096-742acbc7fc3e`) — **success**
2. **Best neighborhoods to stay** (`019fd422-888f-74a3-97ae-0bf1bef830ed`) — **success**
3. **Day trips from Kyoto** (`019fd422-8890-7942-b7bb-5a6cf16011f0`) — **success**

Ready to fetch the results whenever you'd like.

## Collect a result

When a task has succeeded, `check_async_task` returns its final output. We reference the first task (the Kyoto-timing question) by its id.

In [36]:
kyoto_task = list(tasks)[0]

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Give me the findings from task {kyoto_task}."
}]}, config=config)
print_last_exchange(result)

### Human

Give me the findings from task 019fd422-7840-78f1-8096-742acbc7fc3e.

### Ai

## Best Time to Visit Kyoto: Comprehensive Seasonal Guide

### **Peak Season: Late March to Early April (Cherry Blossoms)**

**Best viewing: March 29 - April 5, 2026**

**Pros:**
- Iconic cherry blossoms at full bloom across temples and gardens
- Pleasant mild temperatures (50-60°F)
- Stunning nighttime illuminations on blooming trees (e.g., Kiyomizu Temple)
- Historic temples decorated with sakura create unforgettable scenery

**Cons:**
- Extremely crowded; this is Japan's busiest travel period
- Accommodation costs 2-3x higher than off-season
- Popular temples have queues before opening
- Cherry blossom season is brief (7-10 days at peak)

---

### **Peak Season: Mid-October to Mid-November (Autumn Foliage)**

**Peak foliage: Late October to mid-November**

**Pros:**
- Autumn colors last 1-2 weeks per location (longer viewing window than cherry blossoms)
- Comfortable temperatures (50-65°F); cool but not cold
- Clear, sunny skies predominate
- Gion Odori performances in November
- Rivals spring as Japan's best season
- Festival of the Ages (Jidai Matsuri) on October 22

**Cons:**
- Second busiest season; October 2025 saw record high monthly arrivals (3.9M visitors)
- Accommodation similarly expensive to cherry blossom season
- Popular temples packed; requires early morning visits

---

### **Shoulder Seasons: May & June (Spring Shoulder)**

**Pros:**
- Lush, verdant landscapes post-cherry blossom season
- Aoi Matsuri Festival (May 15) - one of Kyoto's three great festivals
- More moderate crowds than peak seasons
- Comfortable temperatures (60-75°F)

**Cons:**
- Early June marks start of tsuyu (rainy season) with significant rainfall
- Humidity increases through the month

---

### **Summer: July-August (Hot & Humid)**

**Pros:**
- Gion Matsuri Festival (July 1-31) - Japan's largest festival with decorated floats, yoi-yama street festivals, and traditional celebrations
- Kamo River Yuka dining (outdoor riverside dining) in July
- Gozan Okuribi bonfires (August 16) light up five mountains
- Summer discounts available

**Cons:**
- Extremely hot and humid (75-85°F, but feels much hotter with humidity)
- High rainfall and occasional typhoons (late August into early September)
- Air quality can be poor
- Typhoons in September can disrupt transportation
- Peak tourist season despite uncomfortable weather

---

### **Winter: December-February (Cold but Quiet)**

**Pros:**
- **Quietest and most affordable season** - accommodation costs significantly less than peak seasons
- December features seasonal illuminations and Hanatoro light festival
- Hatsumode (New Year shrine visits, Dec 31-Jan 3) with festive atmosphere
- January is described as "genuinely extraordinary" for low crowds
- Clear, mostly sunny days with light precipitation
- Possible snow adds beauty (40-55°F)

**Cons:**
- Cold temperatures (40-55°F); requires winter clothing
- Shorter daylight hours
- December crowding early month due to New Year holidays
- Can look bleak aesthetically

**Best windows:** Mid-January to mid-February and early December

---

### **Late Summer/Early Autumn: September (Transition)**

**Pros:**
- Typhoon season begins to transition toward better weather
- Kamo River Yuka dining
- Lower crowds than summer
- Temperature moderating (70-80°F)

**Cons:**
- Still hot with occasional rain and typhoons
- Transition month; unpredictable weather

---

## **Recommended Itinerary by Priority**

| **Priority** | **Best Months** | **Why** |
|---|---|---|
| **Cherry blossoms + experience crowds** | Late March-Early April | Peak bloom + iconic festival atmosphere |
| **Autumn foliage + fewer crowds** | Early October (before mid-month) | Colors appear earlier, pre-peak crowds |
| **Best weather + value** | May or November | Pleasant temps, manageable crowds |
| **Budget + quiet temples** | January or early December | Lowest prices, fewest visitors |
| **Festivals + culture** | May (Aoi), July (Gion), October (Jidai) | Three great festivals throughout year |

---

## **Verdict**

**Absolute best time:** **Late October to early November** combines superb autumn foliage, pleasant weather (50-65°F), and cultural events with slightly lower crowds than peak cherry blossom season, while foliage views last 1-2 weeks versus days for blossoms.

**Best value:** **January** offers stunning temple experiences with minimal crowds and significantly lower accommodation costs, though you'll need warm clothing.

**Don't miss peak experiences:** Cherry blossoms (late March) and autumn foliage (mid-November) are worth the crowds if seasonal beauty is your priority.

**Avoid:** July-August (heat, humidity, typhoon season) unless specifically attending Gion Matsuri.

## Send follow-up instructions

`update_async_task` interrupts the task's current run and starts a fresh one **on the same thread** — so the subagent sees the full history plus your new message. The **task id stays the same**; only the internal run id changes.

In [37]:
result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Send a follow-up to task {kyoto_task}: also give a rough daily budget "
    "in USD for a mid-range traveler."
}]}, config=config)
print_last_exchange(result)

### Human

Send a follow-up to task 019fd422-7840-78f1-8096-742acbc7fc3e: also give a rough daily budget in USD for a mid-range traveler.

### Ai

Follow-up sent. The task is running again with the new instructions. Let me know when you'd like me to check for the updated results.

In [38]:
# Wait for the updated run, then collect the revised findings.
tasks = supervisor.get_state(config).values.get("async_tasks", {})
wait_all({kyoto_task: tasks[kyoto_task]})

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Show me the updated findings from task {kyoto_task}, including the budget."
}]}, config=config)
print_last_exchange(result)

### Human

Show me the updated findings from task 019fd422-7840-78f1-8096-742acbc7fc3e, including the budget.

### Ai

## Kyoto Daily Budget Breakdown by Season

### Mid-Range Traveler Profile
Based on current 2026 data, a mid-range traveler typically spends **$135-170 USD per day** across Japan. Kyoto runs slightly higher due to temple visits and its popularity.

---

## **Daily Budget Breakdown (Per Person)**

| Category | Off-Season | Shoulder | Peak Season |
|----------|-----------|----------|------------|
| **Accommodation** | $55-75 | $75-100 | $120-180 |
| **Food** | $25-35 | $25-35 | $25-35 |
| **Transportation** | $10-15 | $10-15 | $10-15 |
| **Temples/Attractions** | $15-25 | $15-25 | $15-25 |
| **Miscellaneous** | $10-15 | $10-15 | $10-15 |
| **DAILY TOTAL** | **$115-165** | **$135-190** | **$180-270** |

---

## **Detailed Cost Breakdown by Category**

### **Accommodation**

**Off-Season (January-February, June, December):**
- Business hotel (single room): $55-75/night
- Guesthouse/mid-range hotel: $40-65/night

**Shoulder Season (May, September, early October, late November):**
- Business hotel: $75-100/night
- Guesthouse: $50-80/night

**Peak Season (Late March-Early April, Mid-October to Mid-November):**
- Business hotel: $120-180/night (or higher)
- Mid-range accommodations: $100-150/night
- Average during cherry blossoms: $178/night
- **Note:** Peak season prices can be 2-3x higher than off-season

**Budget options:**
- Capsule hotels/hostels: $25-40/night
- Private hostel dorms: $20-35/night

---

### **Food**

**Daily budget: $25-35/day (consistent across seasons)**

- Gyudon (beef bowl): ~$3.50
- Ramen: ~$4-5
- Convenience store meals: ~$3.50-7
- Casual restaurant lunch: ~$7-10
- Mid-range restaurant dinner: ~$10-17

**Sample daily breakdown:**
- Breakfast: $4-6
- Lunch: $7-10
- Dinner: $12-15
- Snacks/beverages: $3-5

---

### **Transportation**

**Daily cost: $10-15**

- One-day bus pass: $4.70-5.40
- One-day subway pass: ~$5.40
- One-day bus + subway pass: ~$9-10
- Daily IC card usage: $7-13/day

---

### **Temple/Attraction Entrance Fees**

**Daily budget: $15-25**

**Typical temple fees:**
- Most temples: $2-5.50
- Garden entry: $2-2.70
- Larger temples with special exhibits: $4-7

**Free attractions:**
- Fushimi Inari Taisha
- Yasaka Shrine
- Gion district walking
- Arashiyama Bamboo Grove
- Philosopher's Path
- Kyoto Imperial Palace Park

---

### **Miscellaneous Expenses**

**$10-15/day:**
- Souvenirs/gifts: $0-10
- Coffee/drinks outside meals: $3-5
- Vending machine drinks: $1.50-2.50

---

## **Seasonal Cost Comparison (5-Day Trip)**

| Season | Daily Budget | 5-Day Total | Notes |
|--------|-------------|------------|-------|
| **Off-Season** (Jan-Feb, Jun, Dec) | $115-165 | $575-825 | Best value; cheapest accommodation |
| **Shoulder** (May, Sep, early Oct) | $135-190 | $675-950 | Good balance; moderate crowds |
| **Peak** (Late Mar-Apr, Mid-Oct-Nov) | $180-270 | $900-1,350 | Cherry blossoms/foliage; highest prices |

---

## **Sample Daily Itineraries & Costs**

### **Budget Day (Off-Season)**
- Hostel dorm bed: $28
- Street food breakfast + ramen lunch + casual dinner: $18
- Free walking tour: $0
- One paid temple: $5
- City bus all-day pass: $5
- Snacks: $3
- **Total: ~$59-65**

### **Mid-Range Day (Off-Season)**
- Business hotel: $70
- Breakfast + lunch + dinner: $30
- 2-3 temples: $12
- IC card transport: $12
- Coffee + snacks: $6
- **Total: ~$130-140**

### **Mid-Range Day (Peak Season)**
- Business hotel: $150
- Breakfast + lunch + dinner: $30
- 2-3 temples + museum: $18
- IC card transport: $12
- Coffee + snacks: $6
- **Total: ~$216-226**

---

## **Money-Saving Tips**

**Accommodation:**
- Stay slightly outside central Kyoto: save $15-30/night
- Book 2-4 weeks in advance
- Use hostels to cut costs in half

**Food:**
- Eat breakfast at convenience stores: $3-4
- Lunch set meals at casual restaurants: $7-10
- Avoid Gion/tourist-focused restaurants

**Transportation:**
- Walk whenever possible
- Use day passes for 3+ trips per day

**Attractions:**
- Focus on free sites (Fushimi Inari, Arashiyama Bamboo Grove, neighborhood walks)
- Visit 1-2 paid temples daily rather than many

---

## **Bottom Line Daily Budget Estimates**

- **Budget traveler:** $70-100/day
- **Mid-range traveler:** $130-170/day (off-season) / $200-240/day (peak)
- **Luxury traveler:** $300+/day

**For a 7-day trip in mid-range style:**
- Off-season: $910-1,190
- Peak season: $1,400-1,680

*Excludes international flights and long-distance intercity travel.*

## Cancel a task

`cancel_async_task` stops a running job you no longer need.

In [39]:
# Launch one more...
result = supervisor.invoke({"messages": [{"role": "user", "content":
    "Start a background researcher on the best ryokan (traditional inns) in Hakone; "
    "just give me the task id."
}]}, config=config)
print_last_exchange(result)

### Human

Start a background researcher on the best ryokan (traditional inns) in Hakone; just give me the task id.

### Ai

Task ID: `019fd423-ce24-7443-854f-f7af69d48e66`

In [40]:
# ...then cancel it by id.
hakone_task = list(supervisor.get_state(config).values["async_tasks"])[-1]

result = supervisor.invoke({"messages": [{"role": "user", "content":
    f"Cancel task {hakone_task}."
}]}, config=config)
print_last_exchange(result)

print()
tasks = show_tasks()   # assigned, so the clean table prints without echoing the raw dict

### Human

Cancel task 019fd423-ce24-7443-854f-f7af69d48e66.

### Ai

Task cancelled.


019fd422-7840-78f1-8096-742acbc7fc3e  status=success    agent=researcher
019fd422-888f-74a3-97ae-0bf1bef830ed  status=success    agent=researcher
019fd422-8890-7942-b7bb-5a6cf16011f0  status=success    agent=researcher
019fd423-ce24-7443-854f-f7af69d48e66  status=cancelled  agent=researcher


## Recap

- **Async subagents run in the background** on an Agent Protocol server. Launching returns a task id immediately; the supervisor stays responsive.
- Five tools manage the lifecycle: **`start` / `check` / `update` / `cancel` / `list`**.
- Task metadata lives in the **`async_tasks`** state channel, so ids survive context compaction — pair the supervisor with a **checkpointer + `thread_id`**.
- **Transport** is chosen by the spec: **omit `url`** for in-process **ASGI** (co-deployed graphs), **set `url`** for **HTTP** to a remote server.

**Two rules the middleware's system prompt enforces** (they're load-bearing):

1. After launching, return control to the user — never auto-check or poll in a loop.
2. Statuses in the chat history are stale; always `check`/`list` to get the live status before reporting it.

**When to reach for async over the sync `task` tool:** long-running work, jobs that belong on a specialized remote deployment, or many concurrent tasks whose results you collect later. If a task exhausts the worker pool and launches start queuing, raise `--n-jobs-per-worker`.